# ScoutCard AI: run the tracker on a GPU

Runtime → Change runtime type → **T4 GPU** first. Then run the cells top to bottom.

Before this, on your Mac (in the repo folder), make the one file this notebook takes:

```bash
cd ~/scoutai && git pull origin claude/scoutcard-ai-v1-scaffold-c4bgpf
zip -r ~/Desktop/video-service.zip video-service \
  -x "video-service/.venv/*" "*/__pycache__/*" "video-service/tracks/*" \
     "video-service/detections/*" "video-service/debug/*" "video-service/recordings/*"
```

It holds the code, the cropped clips and each clip's calibration `.json`. The clips are game film, so use
your own Google account and delete the runtime when you're done (Runtime → Disconnect and delete runtime).

In [ ]:
!nvidia-smi -L
import torch
print("cuda:", torch.cuda.is_available())

## 1. Upload `video-service.zip` (drag it into the Files panel on the left, or run this cell)

In [ ]:
import os
if not os.path.exists("video-service.zip"):
    from google.colab import files
    files.upload()  # pick video-service.zip
!rm -rf video-service && unzip -q video-service.zip && ls video-service/clips

## 2. Install (the same OpenCV the Mac uses)

In [ ]:
!pip install -q "opencv-python-headless==5.0.0.93" transformers timm pillow pytest
import cv2; print(cv2.__version__)

## 3. Run the tracker\nChange the settings here. `r101vd` + 4 tiles is the biggest; `r50vd` + 3 tiles is what the Mac ran.

In [ ]:
CHECKPOINT = "PekingU/rtdetr_v2_r101vd"   # or rtdetr_v2_r50vd
CONFIDENCE = "0.3"
TILES = 4
SECONDS = 8
import os
os.environ["MODEL_CHECKPOINT"] = CHECKPOINT
os.environ["DETECT_CONFIDENCE"] = CONFIDENCE
%cd video-service
!python -m eval.tracks clips/ --out tracks --seconds {SECONDS} --tiles {TILES} --debug debug/ --detections detections/

## 4. Download the results\nSend the `detections/*.json` files back (they hold numbers, no film). The pictures are in `debug/`.

In [ ]:
!zip -qr results.zip detections tracks debug
from google.colab import files
files.download("results.zip")